# LocalDesk: Qwen decision routing on real IT tickets

An original notebook using the public Unsloth API. Dataset: Leonardo Santiago Benitez Pereira, [Zenodo DOI 10.5281/zenodo.7648117](https://doi.org/10.5281/zenodo.7648117), CC BY 4.0.

Use a fresh Google Colab runtime with an NVIDIA GPU. Availability and free quotas vary. This notebook has been syntax-checked, not executed on a GPU. No measured Qwen results or fixed VRAM guarantee are claimed.

The included CPU baseline was trained separately. This notebook trains the Qwen model through the same configurable service pipeline. Change the task/experiment to reuse it for new labels.

## 1. Download the project and install the GPU recipe

This downloads public code, dependencies and model weights. Repository code is MIT; dependencies and the base model retain their licenses. The two upstream source revisions and base model revision are pinned. Transitive dependencies are recorded after training. Change REPO_REF to a reviewed commit for a frozen experiment.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/denis-samatov/localdesk.git"
REPO_REF = "main"  # Replace with a reviewed commit SHA to freeze the project.
PROJECT = Path("/content/localdesk")
if not PROJECT.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT)], check=True)
subprocess.run(["git", "checkout", REPO_REF], cwd=PROJECT, check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements-gpu.txt"], cwd=PROJECT, check=True
)
subprocess.run([sys.executable, "-m", "pip", "install", "-e", "."], cwd=PROJECT, check=True)

## 2. Prepare verified real data

The script downloads four CSVs from the fixed Zenodo record and verifies SHA-256 checksums. It joins texts and human labels by ID, removes contradictory/preprocessing-equivalent texts and removes training copies of test texts. Calibration comes only from the remaining official training pool.

Six categories have enough unique texts. EOL has only two unique templates and becomes a separate unsupported-category challenge. No action, urgency or Russian labels are fabricated. Expected counts: 690 train, 170 calibration, 524 test, 2 OOD.

In [ ]:
import json

subprocess.run([sys.executable, "scripts/prepare_data.py"], cwd=PROJECT, check=True)

manifest = json.loads((PROJECT / "data/zenodo/manifest.json").read_text())
print(manifest["counts"])
print("Cross-split duplicate groups:", manifest["audit"]["cross_split_groups"])

## 3. Train, calibrate, test, save

Only train rows update model parameters. Calibration rows fit probability temperatures. Test rows are evaluated after training and calibration. The shared configurable pipeline fails if upstream skips or truncates examples. Full run: two epochs; add `--max-steps 20` for a smoke run (not a qualified model). T4 uses float32 rather than pure float16.

No Hugging Face upload or external ticket submission occurs. A checkpoint already at OUT is never overwritten.

In [ ]:
config = json.loads((PROJECT / "experiments/it-support.json").read_text())
config["training"] = {
    "backend": "unsloth",
    "epochs": 2,
    "rank": 16,
    "batch_size": 2,
    "gradient_accumulation": 4,
    "learning_rate": 2e-4,
}
config["registry"] = "../artifacts/qwen-registry"
(PROJECT / "experiments/it-qwen.json").write_text(json.dumps(config, indent=2))
OUT = "artifacts/qwen-registry/qwen-v1"
subprocess.run(
    ["localdesk", "train", "experiments/it-qwen.json", "--run-id", "qwen-v1"],
    cwd=PROJECT,
    check=True,
)

## 4. Inspect actual results and verify saved-checkpoint inference

These values come from your GPU run. Compare with `reports/baseline.json` on the identical cleaned test protocol. Inference is explicitly offline and requires the base weights already cached by training. Failure is an error, with no CPU fallback.

In [ ]:
run = json.loads((PROJECT / OUT / "manifest.json").read_text())
report = json.loads((PROJECT / OUT / "report.json").read_text())
print(
    json.dumps(
        {
            "quality": run["quality"],
            "test": report["classification"],
            "training": report["training"],
        },
        indent=2,
    )
)
if run["quality"]["qualified"]:
    subprocess.run(
        ["localdesk", "predict", "--model", OUT, "File share access request"],
        cwd=PROJECT,
        check=True,
    )
else:
    print("Candidate did not pass gates; inspect the report. It cannot be promoted or served.")

## 5. Keep your artifact

Colab runtimes are temporary. The adapter alone requires the pinned base model in the target computer cache; archive the adapter and arrange a local base-model cache before offline use. Review licenses and runtime-freeze.txt before distributing weights. No automatic model publication is included.

In [ ]:
import shutil

archive = shutil.make_archive(
    str(PROJECT / "artifacts/qwen-localdesk-export"), "zip", PROJECT / OUT
)
print("Saved adapter archive:", archive)
try:
    from google.colab import files
except ImportError:
    print("Outside Colab: copy the archive from the path above.")
else:
    files.download(archive)

## Evaluation limits

The cleaned holdout is from one organization and differs from the original published split after deduplication. It is not a field study in a school/library. The source lacks row-level language labels; Russian performance remains unqualified. Softmax confidence and two OOD examples do not establish general out-of-domain detection. Review thresholds are fixed illustrative settings, not certified operational guarantees.